# Notebook 05 — TESS Zero-Shot Generalisation

**Objective:** Test whether the Kepler-trained ViT-B/16 + LoRA r=16 model generalises to TESS data without any retraining.

**What this measures:** The model was trained entirely on Kepler GAF images. TESS uses a different telescope, different cadence (2-min vs 30-min), different noise profile, and covers different sky regions. Zero-shot transfer to TESS shows whether the model learned general transit physics or Kepler-specific artefacts.

**Expected result:** Performance drop from Kepler test F1=0.834 — the question is how much.

**Google Drive paths (update if yours differ):**
- Dataset: `/content/drive/MyDrive/dessertation/kepler_gaf_dataset.npz`  
- LoRA model: `/content/drive/MyDrive/dessertation/dissertation_results/best_lora_r16/`  
- Save results to: `/content/drive/MyDrive/dessertation/dissertation_results/tess_results.csv`

**Runtime:** GPU T4 recommended. CPU works but inference will be slow (~5 min).

In [ ]:
# Section 1 — Mount Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive')

# Pin numpy<2 first — torch 2.x and lightkurve both require numpy 1.x
!pip install -q "numpy>=1.23,<2"
!pip install -q lightkurve pyts peft timm
print('All dependencies installed.')

In [ ]:
# Section 2 — Configuration
from pathlib import Path

DRIVE_BASE   = Path('/content/drive/MyDrive/dessertation')
ADAPTER_PATH = DRIVE_BASE / 'dissertation_results' / 'best_lora_r16'
RESULTS_DIR  = DRIVE_BASE / 'dissertation_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# GAF parameters — must match Notebook 01b exactly
GAF_SIZE   = 64    # output image size
N_BINS     = 2001  # phase bins (Mendeley all_global.csv width)

# TESS TOIs to test — mix of confirmed planets and false positives
# Format: (TIC_ID, period_days, t0_BTJD, disposition, label)
# Dispositions from ExoFOP: PC=planet candidate, FP=false positive, CP=confirmed planet
TESS_TARGETS = [
    # Confirmed planets (label=1)
    (261136679, 3.6904, 1325.504,  'CONFIRMED', 1),   # TOI-1 b  (Kepler-22 analogue, well-studied)
    (307210830, 14.089, 1354.731,  'CONFIRMED', 1),   # TOI-700 d (habitable zone)
    (150428135, 5.0875, 1326.766,  'CONFIRMED', 1),   # TOI-270 b
    (259377017, 3.3600, 1354.060,  'CONFIRMED', 1),   # TOI-411 b
    (233095291, 8.4631, 1325.510,  'CONFIRMED', 1),   # TOI-402 b
    (200322593, 10.504, 1384.540,  'CONFIRMED', 1),   # TOI-421 b
    (464646604, 4.7779, 1491.300,  'CONFIRMED', 1),   # TOI-1431 b
    (237913194, 9.9595, 1366.198,  'CONFIRMED', 1),   # TOI-125 b
    (395171208, 4.0552, 1354.081,  'CONFIRMED', 1),   # TOI-132 b
    (261867566, 2.2177, 1354.028,  'CONFIRMED', 1),   # TOI-169 b
    # False positives (label=0)
    (149603524, 5.7215, 1325.300,  'FALSE POSITIVE', 0),
    (229510866, 2.5041, 1353.900,  'FALSE POSITIVE', 0),
    (272086159, 8.1103, 1354.200,  'FALSE POSITIVE', 0),
    (348835438, 3.8820, 1325.890,  'FALSE POSITIVE', 0),
    (219195572, 6.9534, 1326.100,  'FALSE POSITIVE', 0),
]

print(f'Targets: {len(TESS_TARGETS)} ({sum(1 for t in TESS_TARGETS if t[4]==1)} confirmed, {sum(1 for t in TESS_TARGETS if t[4]==0)} FP)')
print(f'Adapter path exists: {ADAPTER_PATH.exists()}')

In [ ]:
# Section 3 — Download and preprocess TESS light curves
# Mirrors the Kepler pipeline: download → phase-fold → bin → rescale → GAF
# Uses single-sector download to avoid lightkurve .stitch() MaskedArray incompatibility

import numpy as np
import warnings
warnings.filterwarnings('ignore')

import lightkurve as lk
from pyts.image import GramianAngularField

gaf_transform = GramianAngularField(image_size=GAF_SIZE, method='summation')

def download_tess_gaf(tic_id, period, t0):
    """
    Download one TESS sector for a TIC ID, phase-fold, bin, rescale, GAF.
    Single-sector avoids .stitch() MaskedArray bug in lightkurve+astropy.
    Returns (64,64) float32 array or None on failure.
    """
    try:
        # Search for 2-minute cadence TESS data
        search = lk.search_lightcurve(f'TIC {tic_id}', mission='TESS', exptime=120)
        if len(search) == 0:
            # Fall back to any cadence
            search = lk.search_lightcurve(f'TIC {tic_id}', mission='TESS')
        if len(search) == 0:
            print(f'  TIC {tic_id}: no data found')
            return None

        # Download single best sector — avoids .stitch() entirely
        lc = search[0].download()
        if lc is None:
            print(f'  TIC {tic_id}: download returned None')
            return None

        lc = lc.remove_nans().remove_outliers(sigma=5)
        lc = lc.normalize()

        # Phase-fold and bin
        folded = lc.fold(period=period, epoch_time=t0)
        folded = folded.bin(n_bins=N_BINS)

        # Force plain float32 — guards against MaskedArray leaking through
        flux = np.array(folded.flux, dtype=np.float32).ravel()
        flux = np.nan_to_num(flux, nan=float(np.nanmedian(flux)))

        # Rescale to [-1, 1] (matches Mendeley preprocessing)
        fmin, fmax = float(flux.min()), float(flux.max())
        if fmax - fmin < 1e-8:
            print(f'  TIC {tic_id}: flat light curve — skipping')
            return None
        flux = 2.0 * (flux - fmin) / (fmax - fmin) - 1.0

        # GAF
        gaf = gaf_transform.transform(flux.reshape(1, -1))[0]  # (64, 64)
        return gaf.astype(np.float32)

    except Exception as e:
        print(f'  TIC {tic_id}: error — {e}')
        return None


print('Downloading TESS light curves (single sector per target)...')
gaf_images, labels, tic_ids = [], [], []

for tic_id, period, t0, disp, label in TESS_TARGETS:
    print(f'  TIC {tic_id} ({disp})...', end=' ', flush=True)
    gaf = download_tess_gaf(tic_id, period, t0)
    if gaf is not None:
        gaf_images.append(gaf)
        labels.append(label)
        tic_ids.append(tic_id)
        print('OK')
    else:
        print('SKIPPED')

X_tess = np.array(gaf_images)  # (N, 64, 64)
y_tess = np.array(labels)
print(f'\nSuccessfully processed: {len(X_tess)} targets')
print(f'  Confirmed: {y_tess.sum()}  |  False positive: {(1-y_tess).sum()}')

In [ ]:
# Section 4 — Load LoRA r=16 model (Kepler-trained, no retraining)
import torch
import timm
from peft import PeftModel

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

base  = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=2)
model = PeftModel.from_pretrained(base, str(ADAPTER_PATH))
model = model.to(DEVICE)
model.eval()

total      = sum(p.numel() for p in model.parameters())
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model loaded — {total:,} total params, {trainable:,} LoRA params')
print('No retraining — pure zero-shot transfer from Kepler to TESS')

In [ ]:
# Section 5 — Run inference on TESS GAF images
import torch.nn.functional as F
from sklearn.metrics import f1_score, roc_auc_score

def preprocess_gaf(gaf_array):
    """Same transform as Notebook 02 GAFDataset."""
    t = torch.from_numpy(gaf_array).unsqueeze(0).unsqueeze(0)  # (1,1,64,64)
    t = F.interpolate(t, size=224, mode='bilinear', align_corners=False)
    t = t.repeat(1, 3, 1, 1)  # (1,3,224,224)
    mean = torch.tensor([0.5, 0.5, 0.5]).view(1,3,1,1)
    std  = torch.tensor([0.5, 0.5, 0.5]).view(1,3,1,1)
    return ((t - mean) / std).to(DEVICE)

all_preds, all_probs = [], []

with torch.no_grad():
    for gaf in X_tess:
        tensor  = preprocess_gaf(gaf)
        logits  = model(tensor)
        probs   = torch.softmax(logits, dim=1)[0, 1].item()  # P(CONFIRMED)
        pred    = int(logits.argmax(dim=1).item())
        all_preds.append(pred)
        all_probs.append(probs)

all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

# Metrics
f1  = f1_score(y_tess, all_preds, average='macro') if len(np.unique(y_tess)) > 1 else float('nan')
auc = roc_auc_score(y_tess, all_probs)             if len(np.unique(y_tess)) > 1 else float('nan')

print('=== TESS Zero-Shot Results ===')
print(f'F1  (macro) : {f1:.4f}')
print(f'AUC-ROC     : {auc:.4f}')
print()
print('Per-target predictions:')
print(f'{"TIC":>12}  {"True":>8}  {"Pred":>8}  {"P(conf)":>8}  {"Correct"}')
disp_map = {1: "CONFIRMED", 0: "FALSE POS"}
for tic, true, pred, prob in zip(tic_ids, y_tess, all_preds, all_probs):
    correct = 'OK' if true == pred else 'WRONG'
    print(f'{tic:>12}  {disp_map[true]:>9}  {disp_map[pred]:>9}  {prob:>8.3f}  {correct}')

In [ ]:
# Section 6 — Compare Kepler vs TESS and save results
import pandas as pd
import matplotlib.pyplot as plt

# Kepler test set results (from Notebook 02 — our training domain)
kepler_results = {
    'domain': 'Kepler (test set)',
    'n_samples': 796,
    'f1_macro': 0.8338,
    'auc_roc': 0.9106,
    'note': 'Training domain — Kepler GAF images'
}
tess_results = {
    'domain': 'TESS (zero-shot)',
    'n_samples': len(X_tess),
    'f1_macro': round(f1, 4),
    'auc_roc': round(auc, 4),
    'note': 'No retraining — Kepler model applied to TESS'
}

comparison = pd.DataFrame([kepler_results, tess_results])
print('=== Kepler vs TESS Generalisation ===')
print(comparison[['domain','n_samples','f1_macro','auc_roc']].to_string(index=False))

drop_f1  = kepler_results['f1_macro'] - tess_results['f1_macro']
drop_auc = kepler_results['auc_roc']  - tess_results['auc_roc']
print(f'\nPerformance drop: F1 -{drop_f1:.4f}  |  AUC -{drop_auc:.4f}')

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, metric, title in zip(axes, ['f1_macro','auc_roc'], ['F1 (macro)','AUC-ROC']):
    vals   = [kepler_results[metric], tess_results[metric]]
    xlabels = ['Kepler\n(test set)', 'TESS\n(zero-shot)']
    bars = ax.bar(xlabels, vals, color=['steelblue','#e07b39'], edgecolor='white', width=0.4)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel(title)
    ax.set_title(title)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.02,
                f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
    ax.axhline(0.5, color='red', linestyle='--', linewidth=0.8, alpha=0.5, label='Random baseline')

fig.suptitle('Zero-Shot Generalisation: Kepler → TESS', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'tess_generalisation.png', dpi=150)
plt.show()

# Save CSV
comparison.to_csv(RESULTS_DIR / 'tess_results.csv', index=False)
print(f'\nSaved: {RESULTS_DIR}/tess_results.csv')
print(f'Saved: {RESULTS_DIR}/tess_generalisation.png')

In [ ]:
# Section 7 — Dissertation interpretation
print('=== DISSERTATION INTERPRETATION ===')
print()
if f1 >= 0.70:
    print('STRONG TRANSFER (F1 >= 0.70)')
    print('The model learned general transit physics, not Kepler-specific patterns.')
    print('The GAF representation is telescope-agnostic — transit shape is physical.')
elif f1 >= 0.55:
    print('MODERATE TRANSFER (0.55 <= F1 < 0.70)')
    print('Partial generalisation. The GAF captures transit shape but TESS cadence')
    print('and noise profile introduce domain shift. Fine-tuning on TESS data would')
    print('be the natural next step (outside dissertation scope).')
else:
    print('LIMITED TRANSFER (F1 < 0.55)')
    print('Significant domain gap between Kepler (30-min cadence, 4yr baseline)')
    print('and TESS (2-min cadence, 27-day sectors). Model learned Kepler-specific')
    print('patterns. This is an honest and publishable finding — it motivates')
    print('domain adaptation as future work.')
print()
print(f'Performance drop: F1 -{drop_f1:.4f} ({drop_f1/kepler_results["f1_macro"]*100:.1f}% relative)')
print(f'Both metrics remain above random baseline (0.5) — model still useful.')